# 01. VLA 이해하기 — 보고, 지시를 이해하고, 행동하기

**읽기·토론 60분 · 코드 실행 없이 공부하는 이론 노트북**

VLA는 **영상과 언어 지시를 바탕으로 로봇의 행동을 예측하도록 학습된 모델** 입니다. 이 수업에서는 로봇의 현재 상태도 입력합니다. 중요한 질문은 “사진에 무엇이 있는가?”에서 한 걸음 더 나아간 **“이 상황에서 지시를 수행하려면 로봇을 어떻게 움직여야 하는가?”** 입니다.

이 노트북을 마치면 다음을 자신의 말로 설명할 수 있습니다.

- Vision·Language·Action은 각각 어떤 정보를 뜻하는가?
- VLM의 답변을 제어기에 연결한 4일차 시스템과 VLA는 무엇이 다른가?
- 행동은 어디서 배우며, 실행 중에는 왜 계속 다시 관측하는가?
- 준비된 모델이 있어도 새로운 로봇·작업에서 바로 성공하지 못하는 이유는 무엇인가?

**진행:** 사례·입력 10분 → 4일차 비교·행동 의미 10분 → 학습과 추론 15분 → 행동 묶음·한계 10분 → 모델·수업 연결 10분 → 확인 질문 5분.

이론을 읽는 데 GPU 검사·모델 실행·Python 커널 준비는 필요하지 않습니다. 실행 환경 확인은 [03 컨테이너 준비](03_container.ipynb)에서 진행합니다. [5일차 전체 안내](README.md)

## 1. 같은 장면에서도 지시가 바뀌면 행동이 달라집니다

탁자 위에 **빨간 컵, 파란 컵, 쟁반** 이 있고 로봇팔이 앞에 있다고 생각해 봅시다. 아래는 개념을 설명하기 위한 가상의 예시이며, 배포 모델이 이 장면에서 성공한다는 뜻은 아닙니다.

| 사람이 준 지시 | 로봇이 해결해야 할 일 |
|---|---|
| “빨간 컵을 쟁반에 놓아 줘.” | 빨간 컵에 접근하고, 잡고, 들어 올린 뒤 쟁반으로 옮겨 놓기 |
| “파란 컵을 쟁반에 놓아 줘.” | 같은 영상에서 다른 물체를 선택해 움직이기 |
| “빨간 컵을 오른쪽으로 밀어 줘.” | 같은 물체를 대상으로도 잡아 들기와 다른 접촉 동작 수행하기 |

물체를 찾는 것만으로는 일이 끝나지 않습니다. 그리퍼가 어디에 있는지, 컵을 이미 잡았는지, 어느 방향으로 접근해야 하는지도 행동에 영향을 줍니다. **장면은 현재 상황, 언어는 목표, 로봇 상태는 자신의 현재 자세** 를 알려 줍니다.

VLA는 이런 입력과 행동 사이의 관계를 데이터에서 배웁니다. 위 표의 순서를 반드시 문장 목록으로 먼저 작성한 뒤 실행하는 것은 아닙니다. 많은 VLA는 입력을 받아 바로 로봇 제어에 사용할 수 있는 행동 값이나 행동 묶음을 예측합니다. 대표적인 행동 토큰 방식은 [RT-2의 공식 설명](https://robotics-transformer2.github.io/)에서 확인할 수 있습니다.

## 2. V·L·A를 하나씩 읽어 봅시다

![영상·언어·로봇 상태가 학습된 정책으로 들어가고 행동이 제어기와 환경을 거쳐 새 관측으로 돌아오는 흐름](../교재그림/vla-theory-loop.svg)

| 구분 | 예시 | 행동 결정에 필요한 이유 |
|---|---|---|
| **Vision — 영상** | 외부 카메라의 탁자 영상, 손목 카메라의 그리퍼 주변 영상 | 물체·목표 위치·접촉 주변 상황을 관측하기 위해 |
| **Language — 언어** | “컵을 쟁반에 놓아 줘.” | 지금 수행할 목표를 지정하기 위해 |
| **State — 로봇 상태** | 팔의 자세, 끝단 위치, 그리퍼 벌어짐 | 현재 자세에서 가능한 다음 움직임을 정하기 위해 |
| **Action — 행동** | 끝단 이동·회전·그리퍼 조작 값 | 실제 제어기에 전달할 움직임을 표현하기 위해 |

State는 VLA라는 약자에 별도 글자로 들어 있지 않지만, 여러 모델에서 사용하는 중요한 입력입니다. 모든 VLA가 같은 상태 정보나 같은 수의 카메라를 요구하는 것은 아닙니다. 사용할 모델의 입력 규격을 확인해야 합니다.

그림의 **정책(policy)** 은 관측과 목표를 받아 행동을 정하는 함수입니다. 이 노트북에서 “학습된 정책”이라고 하면, 사람이 모든 상황의 조건문을 작성하는 대신 **데이터를 통해 정해진 모델 파라미터로 행동을 예측한다** 는 뜻입니다. 정책이 현재 관측을 이용해도 목표 달성을 보장하는 것은 아닙니다.

## 3. VLM과 VLA는 출력의 용도가 다릅니다

VLM은 영상과 언어를 연결합니다. 예를 들어 “컵이 어디에 있지?”에 답하거나, 주어진 장면의 상태를 설명할 수 있습니다. VLA는 그 연결을 **로봇 행동 예측** 까지 확장합니다. VLM을 바탕으로 로봇 행동 데이터를 학습하는 접근은 [OpenVLA](https://openvla.github.io/)에도 나타납니다.

| 비교 항목 | VLM을 활용한 장면 판단 | VLA의 로봇 행동 예측 |
|---|---|---|
| 질문 | “무엇이 보이는가? 움직여도 되는가?” | “이 목표를 위해 다음에 어떻게 움직일까?” |
| 출력 예 | 설명 문장, 물체 이름, FOLLOW/STOP JSON | 로봇의 이동·회전·그리퍼 행동 값 |
| 출력 이후 | 별도의 프로그램이 뜻을 해석하고 제어에 연결 | 행동을 정해진 단위로 복원해 제어기에 전달 |
| 학습에서 필요한 연결 | 영상과 언어·판단의 관계 | 관측·지시와 실제 로봇 행동의 관계 |

출력에 숫자나 JSON이 있다고 자동으로 VLA가 되는 것은 아닙니다. **그 숫자가 무슨 행동을 의미하며, 관측에서 그 행동을 정하는 관계를 어떻게 만들었는지** 를 봐야 합니다.

반대로 VLA도 로봇 전체를 혼자 운영하지 않습니다. 영상 전처리, 행동 단위 복원, 제어기, 실행 한계 검사, 통신과 성공 판정 같은 프로그램은 여전히 필요합니다.

## 4. 4일차와 무엇이 달라졌나요?

4일차의 마지막 실습까지 함께 비교해 봅시다. 공통점은 모두 관측을 이용해 움직인다는 것입니다. 차이는 **움직임을 만드는 관계를 사람이 정했는지, 로봇 데이터로 학습했는지** 에 있습니다.

| 실습 | 인지·판단 | 실제 이동을 만드는 부분 |
|---|---|---|
| [규칙 기반 손 추종](../09.Pseudo-VLA/01_ros_gesture_action.ipynb) | MediaPipe 손 좌표와 손 모양 규칙 | 사람이 작성한 영상 좌표→거북이 목표 변환과 추종 제어기 |
| [VLM 손 추종](../09.Pseudo-VLA/03_ros_vlm_action.ipynb) | 손 영상에서 VLM이 FOLLOW/STOP 제안 | 손 좌표로 목표를 만들고 속도를 계산하는 제공 제어기 |
| [YOLO pose→LLM→ROS](../09.Pseudo-VLA/06_ros_pose_llm.ipynb) | 가장 큰 사람의 관절 정보에서 LLM이 요청된 코 등의 좌표 선택 | 선택 좌표를 검증한 뒤 거북이 목표·속도로 변환하는 제공 제어기 |
| **5일차 VLA** | 영상·지시·로봇 상태를 학습된 정책에 입력 | **정책이 로봇팔 행동 묶음을 예측** , 제공 프로그램이 복원·실행 |

4일차에는 “영상 오른쪽 점을 추종하라”는 좌표 연결을 사람이 설계했습니다. 5일차의 집기 작업에서는 접근·자세 변화·그리퍼 조작 등 행동 관계를 준비된 정책이 예측합니다. **VLA에서도 아래쪽 제어기는 남아 있지만, 그 제어기에 보낼 행동 목표를 만드는 방식이 달라집니다.**

이 수업에서는 4일차의 모듈 조합을 Pseudo-VLA라고 부릅니다. 시스템 구조를 구별하기 위한 수업 용어이며, 모든 연구에서 동일하게 사용하는 공식 분류명은 아닙니다.

## 5. Action은 “컵을 잡아”라는 문장일까요?

사람에게는 “컵을 잡아”라는 말이 행동 지시로 충분해 보입니다. 하지만 로봇 제어기는 정해진 형식의 값을 받아야 합니다. 행동 표현은 로봇과 제어기에 따라 달라집니다.

| 표현 방식 | 값의 의미 | 주의할 점 |
|---|---|---|
| 관절 목표 | 각 관절을 어느 각도로 둘지 | 로봇마다 관절 수·축·한계가 다름 |
| 끝단 목표 또는 변화량 | 그리퍼 끝을 어디로 이동·회전할지 | 좌표계·절대값/변화량·단위 확인 필요 |
| 속도·힘 등의 명령 | 얼마나 빠르게 또는 어느 힘으로 움직일지 | 해당 명령을 받는 제어기가 필요 |

**이번 수업의 한 시점 행동은 7개 값** 입니다. 끝단 이동 3개, 회전 3개, 그리퍼 1개로 구성됩니다. 제공된 후처리와 제어 규격을 거친 정규화된 값이며, 곧바로 미터·라디안 단위의 절대 위치라고 읽으면 안 됩니다.

예를 들어 이동 성분 하나가 양수라는 사실만으로 “카메라 화면 오른쪽으로 간다”고 해석할 수 없습니다. 로봇 제어기의 좌표계와 축을 알아야 합니다. 거북이의 영상 좌표와 로봇팔의 행동 값을 그대로 바꿔 끼울 수 없는 이유입니다. 자세한 실습 규격은 [05 관측·행동](05_observation_action.ipynb)에서 확인합니다.

## 6. 행동은 어떻게 배우나요? — 시범과 학습

컵을 옮기는 시범을 수집한다고 생각해 봅시다. 각 시점의 카메라 영상, 지시, 로봇 상태와 **실제로 수행한 행동** 을 짝지어 기록합니다. 그리퍼가 컵에서 멀 때의 행동과 컵을 잡은 뒤의 행동은 다릅니다.

![시범 기록을 사용해 정책을 학습하는 과정과 학습된 체크포인트를 읽어 행동을 예측하는 과정의 구분](../교재그림/vla-theory-learning.svg)

학습에서는 기록된 관측에서 어떤 행동이 나와야 하는지 모델이 배우도록 파라미터를 조정합니다. 이런 설명은 **시범 행동을 모방하는 학습** 의 기본 관점입니다. 모든 VLA가 오직 한 종류의 학습만 사용하는 것은 아니며, 데이터 구성과 학습 목적도 모델마다 다릅니다.

영상·언어로 사전학습한 모델은 물체와 지시를 연결하는 출발점이 될 수 있습니다. 그러나 이미지 설명을 잘한다고 그리퍼를 잘 움직이는 것은 아닙니다. **어떤 관측에서 어떤 로봇 행동이 필요한지 배우는 데이터와 학습 과정** 이 더 필요합니다. 영상·언어 사전학습과 로봇 행동 학습을 연결한 사례는 [RT-2](https://robotics-transformer2.github.io/)와 [π₀](https://www.pi.website/blog/pi0)에 설명되어 있습니다.

**체크포인트(checkpoint)** 는 이렇게 학습한 모델 파라미터와 관련 설정을 저장한 결과입니다. 모델 이름이 같아도 어떤 로봇·카메라·작업 데이터로 학습했는지에 따라 실행 결과가 달라질 수 있습니다.

## 7. 실행 중에는 무엇을 하나요? — 추론과 폐루프

<!-- procedure:lesson-37-641571:start -->
![VLA가 행동한 뒤 다시 보는 과정](../교재그림/vla-theory-loop.svg)

새 관측으로 추론을 반복하는 과정입니다. 실행 중 가중치를 학습하는 것과 구분합니다.
<!-- procedure:lesson-37-641571:end -->


학습이 끝난 체크포인트에 현재 입력을 넣어 행동을 얻는 것을 **추론(inference)** 이라고 합니다. 이번 수업은 이 추론을 사용하며, 실행할 때마다 모델 파라미터를 다시 학습하지 않습니다.

컵 옮기기의 실행 흐름을 개념적으로 풀어 보면 다음과 같습니다.

1. **관측:** 컵·쟁반과 그리퍼의 현재 모습을 봅니다.
2. **행동 예측:** 지시와 현재 상태를 함께 넣어 다음 행동을 얻습니다.
3. **일부 실행:** 예측한 움직임을 제어기가 수행합니다.
4. **재관측:** 컵에 가까워졌는지, 그리퍼 위치가 달라졌는지 새로 봅니다.
5. **새 행동 예측:** 바뀐 상황에 맞춰 다음 행동을 다시 얻습니다.

이 반복이 **폐루프(closed loop)** 입니다. 실행 결과가 다음 판단의 입력으로 돌아옵니다. 처음 찍은 사진 한 장만 보고 끝까지 같은 행동을 실행하면, 물체가 밀리거나 그립이 어긋나도 반영하기 어렵습니다.

다시 본다는 것과 실패를 반드시 복구한다는 것은 다릅니다. 학습 데이터에서 보지 못한 상태에 들어가면 계속 잘못된 행동을 낼 수 있습니다. 또한 **입력을 갱신하는 것과 모델 파라미터를 갱신하는 것은 다릅니다.** 여기서는 입력만 새로 들어옵니다.

## 8. 왜 미래 행동을 묶어서 예측하나요?

접근·회전·그리퍼 조작은 시간에 걸쳐 이어집니다. **행동 묶음(action chunk)** 은 앞으로 여러 시점의 행동을 한 번에 예측한 것입니다. 여러 행동을 묶어 내는 구조는 SmolVLA의 행동 생성 부분에도 사용됩니다. [SmolVLA 공식 설명](https://huggingface.co/blog/smolvla)

![50개 행동 중 앞 10개를 실행하고 새 관측으로 다시 50개를 예측하는 과정](../교재그림/vla-theory-chunk.svg)

이번 수업의 정책은 **50시점×7개 값** 을 예측합니다. 50은 시간 방향의 길이이고, 7은 한 시점의 행동 성분 수입니다. 로봇 50대나 관절 50개라는 뜻이 아닙니다.

예측 길이와 실행 길이는 구분합니다. 수업의 기준 실행은 앞 **10개** 행동만 적용한 뒤 새로 관측합니다. 남은 **40개** 를 이어서 실행하지 않고, 새 관측으로 얻은 새 묶음으로 교체합니다. 이 그림의 50/10은 **현재 교재의 설정** 이며 모든 VLA의 고정 규칙이 아닙니다.

| 실행 길이 | 바뀌는 점 | 함께 생각할 비용 |
|---|---|---|
| 짧게 실행 후 재관측 | 더 자주 새 상황을 반영할 기회 | 정책을 더 자주 호출하는 계산 부담 |
| 길게 실행 후 재관측 | 한 번의 추론 결과로 더 오래 움직임 | 관측 이후 달라진 상황을 늦게 반영할 가능성 |

짧게 설정하면 무조건 성공하는 것도 아닙니다. 추론 속도, 학습된 행동 패턴, 실제 제어 주기와 함께 살펴야 합니다. 이번 제공 실행기는 **추론 중 시뮬레이션 스텝을 진행하지 않는 동기 방식** 입니다. 실제 로봇은 계산 중에도 주변 상황이 변할 수 있으므로, 시뮬레이션 결과를 실물의 반응 시간으로 그대로 해석하면 안 됩니다.

## 9. 모델 안에서는 행동을 어떤 형태로 만들까요?

모든 VLA가 같은 구조는 아닙니다. 우선 **행동을 표현하는 방법** 과 **여러 시점을 묶는 방법** 을 분리해 보면 이해하기 쉽습니다.

| 행동 생성 방법 | 초보자가 이해할 핵심 | 대표적인 예 |
|---|---|---|
| **이산 행동 토큰** | 행동 값을 구간·기호로 표현해 예측한 뒤 제어 값으로 복원 | 원래의 OpenVLA |
| **연속 행동 생성** | 실수로 이루어진 행동을 생성하는 별도 부분을 사용 | SmolVLA, π₀ |

토큰은 모델이 처리하는 기호 단위입니다. 행동 토큰은 일상 언어의 단어와 용도가 다르며, 토큰을 로봇 값으로 되돌리는 규칙이 필요합니다. [OpenVLA 공식 자료](https://openvla.github.io/)

SmolVLA는 영상·언어·상태의 정보를 처리하는 부분과, 그 정보를 이용해 행동 묶음을 만드는 **행동 전문가(action expert)** 를 결합합니다. 행동 전문가는 별도의 사람을 뜻하지 않습니다. 행동 생성에 맞춰 학습된 모델 구성 요소입니다. 이 모델은 **flow matching** 을 사용하며, 입문 단계에서는 “학습된 변환을 반복해 잡음에서 조건에 맞는 연속 행동 묶음을 만드는 방법” 정도로 이해하면 됩니다. [SmolVLA 공식 구조 설명](https://huggingface.co/blog/smolvla)

**행동 묶음은 위 두 행과 나란히 놓을 제3의 생성 방식이 아닙니다.** “값을 어떻게 표현하는가?”와 “몇 시점을 묶어 예측하는가?”는 다른 질문입니다. 같은 계열 이름도 후속 버전에서 구조가 달라질 수 있습니다.

## 10. 학습된 정책도 낯선 상황에서는 실패할 수 있습니다

VLA의 중요한 목표 중 하나는 학습한 경험을 새로운 지시·물체·상황으로 넓혀 쓰는 것입니다. 이를 **일반화(generalization)** 라고 합니다. “언어 지시를 받는다”는 사실만으로 모든 말을 이해하고 모든 일을 수행할 수 있는 것은 아닙니다. 모델이 어떤 환경으로 일반화하는지는 실제로 평가해야 합니다. [OpenVLA의 평가·적응 사례](https://openvla.github.io/)

아래는 특정 모델의 실측 결과가 아니라, 오류 원인을 생각하기 위한 수업 예시입니다.

| 달라진 조건 | 생길 수 있는 문제 | 먼저 확인할 것 |
|---|---|---|
| 카메라 방향·색·배경 | 물체와 그리퍼의 관계를 다르게 해석 | 학습 때와 입력 구성이 맞는지 |
| 물체 크기·모양·마찰 | 접근은 맞아도 잡거나 옮기기 실패 | 시범이 다룬 물체와 접촉 조건 |
| 로봇팔·그리퍼 변경 | 같은 행동 값이 다른 움직임을 만듦 | 상태·행동 차원, 단위, 좌표계 |
| 생소한 지시·작업 | 목표 선택 또는 행동 순서 오류 | 학습된 작업 범위와 지시 표현 |
| 실행 지연·오래된 영상 | 이미 달라진 상황에 늦게 대응 | 관측 시각·추론 시간·실행 길이 |

카메라·몸체·관절·그리퍼 같은 로봇 구성을 **embodiment** 라고 부르기도 합니다. 다른 구성으로 옮길 때에는 정책과 실행 규격의 호환성, 필요한 추가 데이터·학습·평가를 확인해야 합니다. 단순히 파일을 복사하는 문제로 생각하지 않습니다.

## 11. 오늘 사용할 SmolVLA와 수업 전체를 연결해 봅시다

이후 실습은 준비된 **LIBERO 작업용 SmolVLA 체크포인트** 를 사용합니다. 인터넷에서 찾은 임의의 기본 모델에 USB 손 영상을 넣는 구성이 아닙니다. 아래 수치는 [현재 교재의 실행 규격](README.md#모델과-데이터-계약)입니다.

| 요소 | 이번 수업의 구성 | 이론과의 연결 |
|---|---|---|
| 장면 | MuJoCo에서 실행되는 LIBERO 작업 | 로봇 행동의 결과가 바뀌는 환경 |
| 영상 | 가상 카메라 2개, 각각 256×256 RGB | V: 작업 주변의 현재 관측 |
| 언어 | 선택된 과제의 지시 | L: 정책이 수행할 목표 |
| 로봇 상태 | 끝단 위치3·회전 표현3·그리퍼 관절2 = 8개 | 자신의 현재 자세 정보 |
| 정책 출력 | 한 시점 7개 값, 50시점 묶음 | A: 학습된 로봇 행동 예측 |
| 실행 | 앞 10개 적용 후 재관측 | 폐루프와 실행 길이 |
| 작업 성공 | 환경의 과제 성공 조건으로 판정 | 행동 생성과 결과 달성의 구분 |

**모델 이름 → 체크포인트 → 입력·출력 규격 → 실행 환경** 이 함께 맞아야 합니다. SmolVLA라는 이름이 같다고 모든 체크포인트가 현재 과제를 해결하는 것은 아닙니다. 이번 수업에서는 준비된 체크포인트를 **추론만** 하고 추가 학습은 하지 않습니다.

“로드 성공 → 행동 출력 → 실행 완료 → 작업 성공”도 별개입니다. 파일을 읽었지만 GPU 메모리가 부족할 수 있고, 행동 형상이 맞지만 그립에 실패할 수 있습니다. 최대 스텝까지 오류 없이 실행했어도 컵을 놓는 목표를 달성하지 못했다면 작업 성공이 아닙니다.

## 12. 자주 생기는 오해

| 오해 | 이렇게 구분하세요 |
|---|---|
| “VLM이 JSON을 내고 로봇이 움직이면 모두 같은 VLA다.” | 관측에서 실제 행동을 만드는 부분이 학습된 정책인지, 사람이 정한 변환인지 확인합니다. |
| “VLA가 모든 모터를 직접 제어하므로 제어기는 필요 없다.” | 출력 행동을 해석하고 적용하는 제어기와 실행 프로그램이 필요합니다. |
| “명령을 바꾸면 어떤 새 작업이든 할 수 있다.” | 학습한 지시·작업·로봇 범위와 일반화 한계를 확인해야 합니다. |
| “다시 관측하니 실행할수록 자동으로 학습된다.” | 재관측은 입력 갱신입니다. 이번 추론 과정은 파라미터를 학습하지 않습니다. |
| “50개를 예측했으니 50개를 모두 실행해야 한다.” | 예측 길이와 실행 길이를 구분합니다. 현재 기준은 50개 예측·10개 실행입니다. |
| “모델이 작거나 로드됐으니 실시간 작업도 성공한다.” | 메모리, 추론 지연, 제어 주기, 실제 성공을 따로 확인합니다. |

## 13. 이해 확인 — 설명해 보고 해설과 비교하세요

코드를 작성하지 않습니다. 말로 설명하거나 Markdown 셀에 짧게 메모하세요. 아래 다섯 질문에 답하면 이론 단원을 마친 것입니다.

1. 같은 장면에서 “빨간 컵” 대신 “파란 컵”을 지시하면 어떤 입력이 달라지나요? 성공이 보장되나요?
2. 4일차 pose→LLM 실습과 5일차 VLA에서 실제 이동을 만드는 부분은 어떻게 다른가요?
3. 새 영상을 받아 행동을 다시 예측하는 것이 왜 추가 학습은 아닌가요?
4. 50개 행동을 예측하고 10개를 실행했다면 나머지 40개는 어떻게 되나요?
5. 모델이 로드되고 50×7 배열이 나왔다면, 아직 무엇을 확인해야 하나요?

<details>
<summary>해설 보기 — 답뿐 아니라 이유까지 확인</summary>

**1. 언어 입력이 달라집니다.** 영상과 로봇 상태가 같아도 목표가 달라졌으므로 적절한 행동은 달라질 수 있습니다. 하지만 색 구분·대상 선택·작업 수행을 모델이 충분히 학습했는지는 별도 문제이므로 성공을 보장하지 않습니다.

**2. 4일차에는 좌표를 목표·속도로 변환하는 관계를 사람이 작성했습니다.** LLM은 관절 좌표 선택에 참여하고 검증된 좌표만 사용했습니다. 5일차에는 준비된 정책이 영상·지시·로봇 상태에서 행동 묶음을 예측합니다. 그 뒤에 행동을 복원하고 적용하는 프로그램·제어기는 여전히 남아 있습니다.

**3. 입력이 바뀌는 것과 파라미터가 바뀌는 것은 다릅니다.** 추론은 학습된 파라미터를 이용해 새 입력의 행동을 계산합니다. 추가 학습에는 학습 목표와 데이터에 따른 파라미터 갱신 과정이 필요하며, 이번 실행은 그 과정을 수행하지 않습니다.

**4. 현재 실행기는 남은 40개를 이어 쓰지 않습니다.** 실행 후의 새 관측으로 다시 50개를 예측하고 그 앞부분을 적용합니다. 물체가 예상과 다르게 움직였을 때 다음 행동을 바꿀 기회를 얻지만, 정책이 올바르게 복구할지는 따로 확인해야 합니다.

**5. 입력·출력의 의미, 실행 가능성, 시간, 과제 성공을 확인해야 합니다.** 배열 크기만 맞아도 단위·축·정규화가 틀리면 잘못 움직일 수 있습니다. 실제 rollout의 영상과 환경 성공 판정도 확인해야 합니다. 실행 완료와 작업 성공을 구분해 기록합니다.

</details>

## 14. 다음 단원과 슬라이드 복습

지금 설명할 핵심 흐름은 **“현재를 관측하고, 언어로 목표를 정하고, 학습된 정책이 행동을 예측하고, 실행 결과를 다시 관측한다”** 입니다. 이 과정에서 행동의 의미와 실행 결과를 사람이 확인해야 합니다.

[02 시뮬레이션 구성](02_simulation_stack.ipynb)에서는 MuJoCo·LIBERO·LeRobot·SmolVLA가 이 흐름의 어느 부분을 맡는지 배웁니다. 환경 준비는 [03](03_container.ipynb), 실제 관측·행동은 [04](04_environment.ipynb)·[05](05_observation_action.ipynb), 추론·폐루프·평가는 [06](06_inference.ipynb)·[07](07_rollout.ipynb)·[08](08_evaluation.ipynb)로 이어집니다.

<!-- slide-sync:start -->
**슬라이드 복습:** [5일차 7쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s003) · [배포 PDF 7쪽](../슬라이드/5일차.pdf#page=7)에서 **7–14쪽** 을 보며 입력·행동 결정·모델과 체크포인트의 차이를 설명합니다. 이 노트북의 확인 질문은 구두 또는 글로 답합니다.
<!-- slide-sync:end -->

그림은 이 교재를 위해 작성한 개념도입니다. 특정 모델의 층 구성·속도·성공률을 측정한 그림은 아닙니다. 심화 자료는 본문의 [RT-2](https://robotics-transformer2.github.io/), [OpenVLA](https://openvla.github.io/), [SmolVLA](https://huggingface.co/blog/smolvla), [π₀](https://www.pi.website/blog/pi0) 공식 설명을 참고하세요.